In [1]:
!pip install -U mljar-supervised

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 86.7 MB/s eta 0:00:00
  Created wheel for mljar-supervised: filename=mljar_supervised-1.1.17-py3-none-any.whl size=164663 sha256=8e94c16ac93ffa4fe3fde6979484e1b069be491bab249dff2f8358395398881c
  Stored in directory: /root/.cache/pip/wheels/6c/65/96/b05fb5e270b0c3cf376990dc459932cd056e6c179467699a81
  Created wheel for mljar-scikit-plot: filename=mljar_scikit_plot-0.3.12-py3-none-any.whl size=32014 sha256=b4448be4928a617e8878de24c7edfa585a781b012d40fca4a5ccfe0c9a6a4a73
  Stored in directory: /root/.cache/pip/wheels/e2/2d/3e/8afe0632e7b03c3ae7b8048f88f0dcca4355b32d31abaea3ba
Successfully built mljar-supervised mljar-

In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [3]:
from supervised.automl import AutoML

In [4]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [5]:
train_x = pandas.read_csv('/kaggle/input/crypto-models/data/X_train.csv')
train_y = pandas.read_csv('/kaggle/input/crypto-models/data/y_train.csv')
test_x = pandas.read_csv('/kaggle/input/crypto-models/data/X_test.csv')
test_y = pandas.read_csv('/kaggle/input/crypto-models/data/y_test.csv')
test = pandas.read_csv('/kaggle/input/drw-crypto-data/test.csv')

In [6]:
automl = AutoML(
    mode="Compete", 
    eval_metric="rmse",
    model_time_limit=5*60,
    total_time_limit=60*60,
    top_models_to_improve=3,
    start_random_models=5,
    hill_climbing_steps=0,
    features_selection=True,
    golden_features=False,
    stack_models=True,
    train_ensemble=True,
    validation_strategy={
        "validation_type": "kfold",
        "k_folds": 3,
        "shuffle": True,
        "stratify": True,
    },
    algorithms=['Random Forest', 
                'Extra Trees', 
                'Xgboost', 
                'CatBoost', 
                'Neural Network', 
                'Nearest Neighbors']
)

automl.fit(train_x, train_y)

AutoML directory: AutoML_1
The task is regression with evaluation metric rmse
AutoML will use algorithms: ['Random Forest', 'Extra Trees', 'Xgboost', 'CatBoost', 'Neural Network', 'Nearest Neighbors']
AutoML will stack models
AutoML will ensemble available models
AutoML steps: ['simple_algorithms', 'default_algorithms', 'not_so_random', 'kmeans_features', 'insert_random_feature', 'features_selection', 'boost_on_errors', 'ensemble', 'stack', 'ensemble_stacked']
Skip simple_algorithms because no parameters were generated.
* Step default_algorithms will try to check up to 5 models
1_Default_Xgboost rmse 0.195835 trained in 2855.02 seconds
2_Default_CatBoost rmse 0.55583 trained in 181.65 seconds
3_Default_NeuralNetwork rmse 1.007269 trained in 42.98 seconds
4_Default_RandomForest rmse 0.978186 trained in 1668.56 seconds
5_Default_ExtraTrees rmse 0.981562 trained in 111.1 seconds
Skip not_so_random because of the time limit.
Skip kmeans_features because of the time limit.
Not enough time t

AutoML(algorithms=['Random Forest', 'Extra Trees', 'Xgboost', 'CatBoost',
                   'Neural Network', 'Nearest Neighbors'],
       eval_metric='rmse', features_selection=True, golden_features=False,
       hill_climbing_steps=0, mode='Compete', model_time_limit=300,
       stack_models=True, start_random_models=5, top_models_to_improve=3,
       validation_strategy={'k_folds': 3, 'shuffle': True, 'stratify': True,
                            'validation_type': 'kfold'})

In [7]:
test_predictions = automl.predict(test)
test_predictions

array([-0.06208736,  0.26016557, -0.21141887, ..., -0.09643576,
       -0.21546419,  0.8016321 ], dtype=float32)

In [8]:
submission = pandas.DataFrame({
    'ID': [i for i in range(1, len(test)+1)],
    'prediction': test_predictions,
})
submission

,ID,prediction
0,1,-0.062087
1,2,0.260166
2,3,-0.211419
3,4,-1.886616
4,5,0.262560
...,...,...
538145,538146,0.992804
538146,538147,-0.492318
538147,538148,-0.096436
538148,538149,-0.215464


In [9]:
submission.to_csv('mljarautoml.csv', index = False)